# Learning in Public: Building a RAG Bot for LLM Zoomcamp


As part of my journey through the DataTalks.Club LLM Zoomcamp, I'm practicing the "learning in public" approach. Instead of keeping my notes tucked away in a local folder, I'm sharing my execution steps, code snippets, and mental models as I build out our running project: a question-answering bot tailored specifically for the course itself. Here is a breakdown of my first hands-on steps with LLMs and the core mechanics of Retrieval-Augmented Generation (RAG).

In this course, our running project is to build a bot that can answer queries related to the course.

DataTalks.Club runs free ZoomCamps on data engineering, machine learning, MLOps, and other topics. Each course has its own FAQ document containing common questions and answers. Many of these questions overlap across different tracks, but they might be worded differently, or the responses might be structured differently. Additionally, some questions are highly course-specific.

We need to build a bot that can ingest all of this knowledge and provide accurate answers in natural language.

## RAG: Retrieval-Augmented Generation

*Why we need RAG?*

LLMs are trained on billions of parameters and capture a    massive footprint of the internet's public knowledge. So, why not just ask the LLM a question directly and be done with it?

Let's test this by building a simple function that takes a question, passes it to an LLM (gpt-5.4-mini in this case), and returns the response.


In [9]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()
import os

# Reuse the existing client if it was already created in another cell
if "openai_client" not in globals():
    openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

Let's see how it goes:

In [12]:
llm("Hey Mr. LLM, how is life treating you?")

'Pretty well, thanks for asking — I’m here and ready to help.\n\nHow’s life treating you?'

Now, let's see how it does with course specific questions.

In [5]:
question = "I just discovered the course. Can I still join?"
llm_response = llm(question)
print(f"ResourceWarning: {llm_response}")


A few quick things to check:
- **Registration deadline**: some courses allow late enrollment, some don’t.
- **Capacity**: there may still be spots available.
- **Prerequisites**: make sure you meet any required background or approval.
- **Access to materials**: if the course has already started, ask whether you’ll get access to recordings, notes, or assignments from the beginning.

If you want, I can help you write a short message to the instructor like:

> Hi [Name], I just discovered your course and I’m very interested in joining. Is it still possible to enroll at this stage? Thank you!

If you share the course name or platform, I can help you figure out the next step.


It's evident that the LLMs would rarely say "I don't know the answer to your question". Here also LLM has given a vague answer. If we ask it about "How to make Biryani?" which is a common knowledge question (Although, making a good Biryani is still very difficult :-), LLMs can fairly answer this question. But they do not know much about LLM-Zoomcamp and they give generic answers. 

## Why is that??
This is because the LLM model is trained on the intelligence openly available on the internet. But there is always some cutoff to the training data's recency. For example, the training data for some LLM is fixed to 30th April 2026, and some event happened on 1st may 2026; the LLM would have no clue about that.

## Adding context manually

Adding context can fix this short-coming of LLMs. LLM zoomcamp website have a lot of FAQs. We get some answers from there and add that to context.

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.


## RAG
RAG stands for *R*etrival *A*ugmented *G*eneration. We retrieve (search) relevant information from documents in our Knowledge base and then augment LLM to generate a response. That search step is what gives the LLM the context it needs to answer correctly.
So, what's happening here is the *Retrival of Information and Generation*.


That's the entire architecture. It comes down to three components.

The pieces are search, the prompt, and the LLM:

- search
- prompt
- LLM

```mermaid
flowchart TD
    U([User])
    APP[Application]
    DB[(Knowledge base)]
    DOCS[[D1 ... D5]]
    PROMPT[Build Prompt<br/>Question + Context]
    LLM[LLM]
    ANSWER([Answer])

    U -->|Question| APP
    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP
    APP --> PROMPT
    PROMPT --> LLM
    LLM --> ANSWER
    ANSWER --> U
```

The LLM only see the data we provide to it; so it's answers are grounded to our data. If right data is retrieved, our answers will be better. if wrong data is retrieved, our answers will not be accurate. Our model is only as good as our retrieval. So search quality matters a lot for RAG.

## Search - The First component
At the core, every search engine does same thing,
- Take a query.
- Evaluate similarity between query and the document and score every document.
- Return the top results

So, we can consider the *Search* as a similarity function. What differentiate every search engine is "What similarity metric is used?" and "How similarity is calculated?"

Broadly, there can be 2 kind of similarity:
- Text/Lexical search: Counts how many words the query and the document share. It looks at the similarity at only at the surface level.
- Vector/Semantic search : In this case similarity compares the meaning of the documents and the query. So this goes at the deeper level to asses the similrity.

Consider these two questions:

"Can I still join the course after the start date?"
"Is it possible to enroll late?"

They mean the same thing, but share almost no keywords. "Join" is not "enroll", "course" is absent, "start date" is not "late". A text search engine would struggle to match them, because it only sees words.

The Vector search is covered later in the course, so for the purpose of this Write-up Text-seach is used. We implement text-search using **minseach** a light weight search engine that can be used with python implementation.


### Indexing with **minsearch**

It is very important to search the documents and index them based on similarity. If we don't index and send all the documents to LLM, it would be very expansive and slow.
[minsearch](https://github.com/alexeygrigorev/minsearch) is a simple in-memory search engine. It's lightweight, so it runs anywhere Python runs. It's a toy implementation, not production ready, but it illustrates how search engines work and it gives good results.

We'll index the question, section, and answer fields as text (they'll be tokenized and ranked), and the course field as a keyword (for filtering).


In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

### Trying a search
Let's try a search with the question we used before:

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

### Boosting fields
Not all fields are equally important. The question field is usually more relevant than section for matching. Query words appearing in the question is a stronger signal than them appearing in the section name.

minsearch supports field boosting to reflect this:

```python
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)
```

### Filtering by course
We want to restrict the search to a specific course.

minsearch supports keyword filtering:

```python
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)
```

### Wrapping it in a function
Let's wrap the search in a search function - the first component of our RAG pipeline:

```python
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )


search_results = search(question)
```

## Building the Prompt
With **Search**, we have indexed our documents and identified the relevant information for 

+:++:

:::::::;    